# Conv-AE baseline, 4-fold LOTO (Table 5 Conv-AE row)

1D convolutional autoencoder on raw current with fixed-length sequences and MSE reconstruction loss.

**Paper:** Zafar SA, Qin W. *Cross-task anomaly detection in reconfigurable industrial robot systems based on physics-structured regression of joint motor currents* (2026).


In [ ]:
import sys
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__} -- {'GPU' if torch.cuda.is_available() else 'CPU'}")
except ImportError:
    print("ERROR: PyTorch not found. Install: pip install torch")
    sys.exit(1)

import os, glob, warnings, time
import numpy as np
import pandas as pd
import h5py
import scipy.stats as sst
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)

ROOT = r"D:\Research\RCM"
BASE = os.path.join(ROOT, "Lab_Data")
OUT  = os.path.join(ROOT, "Processed_Data")
os.makedirs(OUT, exist_ok=True)

# CONSTANTS
TASKS        = ["T1", "T2", "T3", "T4"]
TASK_PAYLOAD = {"T1": 0.0, "T2": 1.0, "T3": 3.0, "T4": 2.0}
RATE         = 125
SUBSAMPLE    = 4
MIN_SAMP     = 200
N_BOOT       = 10000
EPOCHS       = 60
BATCH_SIZE   = 32
LR           = 1e-3
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# REGISTRY
REGISTRY = {
    "T1_healthy":    ("T1_PickPlace/Healthy",  "UR5_T1_healthy_180cyc_*.h5",
                      "T1","healthy","none"),
    "T2_healthy":    ("T2_Assembly/Healthy",    "UR5_T2_healthy_180cyc_*.h5",
                      "T2","healthy","none"),
    "T3_healthy":    ("T3_Palletize/Healthy",   "UR5_T3_healthy_183cyc_*.h5",
                      "T3","healthy","none"),
    "T1_A2_0.5kg":   ("T1_PickPlace/A2","UR5_T1_A2_0.5kg_gripper_40cyc_*.h5",
                      "T1","A2","0.5kg"),
    "T1_A2_1kg":     ("T1_PickPlace/A2","UR5_T1_A2_1kg_gripper_40cyc_*.h5",
                      "T1","A2","1kg"),
    "T1_A2_2kg":     ("T1_PickPlace/A2","UR5_T1_A2_2kg_gripper_40cyc_*.h5",
                      "T1","A2","2kg"),
    "T1_A3_10wraps": ("T1_PickPlace/A3","UR5_T1_A3_1band_40cyc_*.h5",
                      "T1","A3","10wraps"),
    "T1_A3_17wraps": ("T1_PickPlace/A3","UR5_T1_A3_3bands_40cyc_*.h5",
                      "T1","A3","17wraps"),
    "T1_A5_20mm":    ("T1_PickPlace/A5","UR5_T1_A5_20mm_40cyc_*.h5",
                      "T1","A5","20mm"),
    "T1_A5_50mm":    ("T1_PickPlace/A5","UR5_T1_A5_50mm_40cyc_*.h5",
                      "T1","A5","50mm"),
    "T1_A5_100mm":   ("T1_PickPlace/A5","UR5_T1_A5_100mm_40cyc_*.h5",
                      "T1","A5","100mm"),
    "T2_A2_1.5kg":   ("T2_Assembly/A2","UR5_T2_A2_1.5kg_gripper_40cyc_*.h5",
                      "T2","A2","1.5kg"),
    "T2_A2_2kg":     ("T2_Assembly/A2","UR5_T2_A2_2kg_gripper_40cyc_*.h5",
                      "T2","A2","2kg"),
    "T2_A2_3kg":     ("T2_Assembly/A2","UR5_T2_A2_3kg_gripper_40cyc_*.h5",
                      "T2","A2","3kg"),
    "T2_A3_7duct":   ("T2_Assembly/A3","UR5_T2_A3_light_duct_40cyc_*.h5",
                      "T2","A3","7duct"),
    "T2_A3_14duct":  ("T2_Assembly/A3","UR5_T2_A3_medium_duct_40cyc_*_225508.h5",
                      "T2","A3","14duct"),
    "T2_A5_20mm":    ("T2_Assembly/A5","UR5_T2_A5_20mm_40cyc_*.h5",
                      "T2","A5","20mm"),
    "T2_A5_50mm":    ("T2_Assembly/A5","UR5_T2_A5_50mm_40cyc_*.h5",
                      "T2","A5","50mm"),
    "T2_A5_100mm":   ("T2_Assembly/A5","UR5_T2_A5_100mm_40cyc_*.h5",
                      "T2","A5","100mm"),
    "T3_A2_3.5kg":   ("T3_Palletize/A2","UR5_T3_A2_3.5kg_gripper_33cyc_*.h5",
                      "T3","A2","3.5kg"),
    "T3_A2_4kg":     ("T3_Palletize/A2","UR5_T3_A2_4kg_gripper_33cyc_*.h5",
                      "T3","A2","4kg"),
    "T3_A2_5kg":     ("T3_Palletize/A2","UR5_T3_A2_4.5kg_gripper_33cyc_*.h5",
                      "T3","A2","5kg"),
    "T3_A3_7duct":   ("T3_Palletize/A3","UR5_T3_A3_light_duct_33cyc_*_222457.h5",
                      "T3","A3","7duct"),
    "T3_A3_14duct":  ("T3_Palletize/A3","UR5_T3_A3_medium_duct_33cyc_*_205648.h5",
                      "T3","A3","14duct"),
    "T3_A5_20mm":    ("T3_Palletize/A5","UR5_T3_A5_20mm_33cyc_*_172334.h5",
                      "T3","A5","20mm"),
    "T3_A5_50mm":    ("T3_Palletize/A5","UR5_T3_A5_50mm_33cyc_*_164447.h5",
                      "T3","A5","50mm"),
    "T3_A5_100mm":   ("T3_Palletize/A5","UR5_T3_A5_100mm_33cyc_*_160716.h5",
                      "T3","A5","100mm"),
    # T4 entries
    "T4_healthy":    ("T4_BinReorient/healthy", "UR5_T4_healthy_session2_35cyc_*.h5",
                      "T4","healthy","none"),
    "T4_A2_0.5kg":   ("T4_BinReorient/anomaly", "UR5_T4_A2_0.5kg_35cyc_*.h5",
                      "T4","A2","0.5kg"),
    "T4_A2_1kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_1kg_35cyc_*.h5",
                      "T4","A2","1kg"),
    "T4_A2_2kg":     ("T4_BinReorient/anomaly", "UR5_T4_A2_2kg_35cyc_*.h5",
                      "T4","A2","2kg"),
    "T4_A3_7duct":   ("T4_BinReorient/anomaly", "UR5_T4_A3_7wraps_35cyc_*.h5",
                      "T4","A3","7duct"),
    "T4_A3_14duct":  ("T4_BinReorient/anomaly", "UR5_T4_A3_14wraps_35cyc_*.h5",
                      "T4","A3","14duct"),
    "T4_A5_20mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_20mm_35cyc_*.h5",
                      "T4","A5","20mm"),
    "T4_A5_50mm":    ("T4_BinReorient/anomaly", "UR5_T4_A5_50mm_35cyc_*.h5",
                      "T4","A5","50mm"),
    "T4_A5_100mm":   ("T4_BinReorient/anomaly", "UR5_T4_A5_100mm_35cyc_*.h5",
                      "T4","A5","100mm"),
}

# STATISTICAL FUNCTIONS

def bootstrap_auroc_bca(y_true, y_score, n_boot=N_BOOT, rng=None):
    if rng is None:
        rng = np.random.default_rng(42)
    n = len(y_true)
    auroc_obs = roc_auc_score(y_true, y_score)
    boot = np.zeros(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]; ys = y_score[idx]
        if yt.sum() == 0 or yt.sum() == n:
            boot[b] = auroc_obs
        else:
            boot[b] = roc_auc_score(yt, ys)
    prop = np.clip(np.mean(boot < auroc_obs), 1e-6, 1 - 1e-6)
    z0   = sst.norm.ppf(prop)
    jack = np.zeros(n)
    for i in range(n):
        idx_j = np.concatenate([np.arange(i), np.arange(i + 1, n)])
        yt_j  = y_true[idx_j]; ys_j = y_score[idx_j]
        jack[i] = roc_auc_score(yt_j, ys_j) \
            if (0 < yt_j.sum() < len(yt_j)) else auroc_obs
    jm  = jack.mean()
    num = np.sum((jm - jack) ** 3)
    den = 6.0 * (np.sum((jm - jack) ** 2) ** 1.5)
    a   = num / den if den != 0 else 0.0
    ci_lo = ci_hi = None
    for z_a, attr in [(sst.norm.ppf(0.025), "lo"),
                      (sst.norm.ppf(0.975), "hi")]:
        p = sst.norm.cdf(z0 + (z0 + z_a) / (1 - a * (z0 + z_a)))
        p = np.clip(p, 0.001, 0.999)
        if attr == "lo":
            ci_lo = float(np.quantile(boot, p))
        else:
            ci_hi = float(np.quantile(boot, p))
    return float(auroc_obs), ci_lo, ci_hi

# CONV-AE ARCHITECTURE
class ConvAutoencoder(nn.Module):
    def __init__(self, fixed_len: int, n_channels: int = 6):
        super().__init__()
        self.fixed_len  = fixed_len
        self.n_channels = n_channels
        self.encoder = nn.Sequential(
            nn.Conv1d(n_channels, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(32, 16, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(16, 8, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(8, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose1d(16, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv1d(32, n_channels, kernel_size=7, padding=3),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        if x_hat.shape[-1] != x.shape[-1]:
            x_hat = x_hat[..., :x.shape[-1]]
        return x_hat

    def reconstruction_score(self, x):
        with torch.no_grad():
            x_hat = self.forward(x)
            mse = ((x - x_hat) ** 2).mean(dim=[1, 2])
        return mse.cpu().numpy()

# STEP 1 -- Load data
print("=" * 65)
print("NB10bd -- Conv-AE Baseline (4-fold LOTO with T4)")
print("=" * 65)
print(f"\nDevice: {DEVICE}")
print("\n[Step 1] Loading data...")

all_cycles = []
for key, (subdir, pattern, task, anomaly, severity) in REGISTRY.items():
    matches = sorted(glob.glob(os.path.join(BASE, subdir, pattern)))
    if not matches:
        print(f"  WARNING  Not found: {key}")
        continue
    with h5py.File(matches[0], "r") as f:
        cnum    = f["cycle_number"][:].astype(int).ravel()
        cur_all = f["actual_current"][:]
    is_anom = 0 if anomaly == "healthy" else 1
    for c in np.unique(cnum[cnum > 0]):
        mask = cnum == c
        if mask.sum() >= MIN_SAMP:
            all_cycles.append({
                "current":    cur_all[mask],
                "task":       task, "anomaly":    anomaly,
                "severity":   severity, "is_anomaly": is_anom,
            })

healthy_cycles = [c for c in all_cycles if c["is_anomaly"] == 0]
print(f"  Total: {len(all_cycles)} | Healthy: {len(healthy_cycles)}")
for t in TASKS:
    nh = sum(1 for c in all_cycles if c["task"]==t and c["is_anomaly"]==0)
    na = sum(1 for c in all_cycles if c["task"]==t and c["is_anomaly"]==1)
    print(f"    {t}: {nh} healthy, {na} anomaly")

# STEP 2 -- Determine FIXED_LEN
print("\n[Step 2] Determining FIXED_LEN...")
sub_lengths = np.array([len(range(0, len(c["current"]), SUBSAMPLE))
                        for c in all_cycles])
p5  = int(np.percentile(sub_lengths, 5))
p50 = int(np.percentile(sub_lengths, 50))
print(f"  Subsampled lengths: min={sub_lengths.min()}  p5={p5}  "
      f"median={p50}  max={sub_lengths.max()}")
FIXED_LEN = max(p5 - (p5 % 4), 64)
print(f"  FIXED_LEN={FIXED_LEN}")
n_padded    = int(np.sum(sub_lengths < FIXED_LEN))
n_truncated = int(np.sum(sub_lengths > FIXED_LEN))
print(f"  Padded: {n_padded}/{len(all_cycles)}  Truncated: {n_truncated}/{len(all_cycles)}")

def cycle_to_tensor(cyc, fixed_len=FIXED_LEN):
    cur = cyc["current"]
    idx = list(range(0, len(cur), SUBSAMPLE))
    ts  = cur[idx]
    mu  = ts.mean(0, keepdims=True); sg = ts.std(0, keepdims=True) + 1e-8
    ts  = (ts - mu) / sg
    if len(ts) >= fixed_len:
        ts = ts[:fixed_len]
    else:
        pad = np.zeros((fixed_len - len(ts), ts.shape[1]))
        ts  = np.vstack([ts, pad])
    return ts.T.astype(np.float32)

# STEP 3 -- LOTO training and scoring
print("\n[Step 3] LOTO training and scoring...")

agg_scores  = {}
anom_scores = {}

for test_task in TASKS:
    tr_tasks = [t for t in TASKS if t != test_task]
    print(f"\n  Fold: test={test_task}  train={tr_tasks}")

    tr_tensors = [cycle_to_tensor(c)
                  for c in healthy_cycles if c["task"] in tr_tasks]
    X_tr = torch.tensor(np.stack(tr_tensors)).to(DEVICE)
    tr_ds = TensorDataset(X_tr)
    tr_dl = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       drop_last=False)
    print(f"  Training samples: {len(X_tr)}")

    model = ConvAutoencoder(FIXED_LEN).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=LR)
    criterion = nn.MSELoss()

    t0 = time.perf_counter()
    model.train()
    for epoch in range(EPOCHS):
        epoch_loss = 0.0
        for (xb,) in tr_dl:
            opt.zero_grad()
            loss = criterion(model(xb), xb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item() * len(xb)
        epoch_loss /= len(X_tr)
        if (epoch + 1) % 10 == 0:
            print(f"    Epoch {epoch+1:3d}/{EPOCHS}  loss={epoch_loss:.6f}")
    train_s = time.perf_counter() - t0
    print(f"  Training time: {train_s:.1f} s")

    model.eval()
    te_cycles = [c for c in all_cycles if c["task"] == test_task]
    scores, labels, anomalies = [], [], []
    for cyc in te_cycles:
        x = torch.tensor(cycle_to_tensor(cyc)).unsqueeze(0).to(DEVICE)
        scores.append(model.reconstruction_score(x)[0])
        labels.append(cyc["is_anomaly"])
        anomalies.append(cyc["anomaly"])

    y_true  = np.array(labels,  dtype=int)
    y_score = np.array(scores,  dtype=float)
    auroc   = roc_auc_score(y_true, y_score)
    print(f"  Aggregate AUROC ({test_task}): {auroc:.4f}")
    agg_scores[test_task] = (y_true, y_score)

    h_scores = y_score[y_true == 0]
    for anom in ["A2", "A3", "A5"]:
        a_mask = (np.array(anomalies) == anom) & (y_true == 1)
        if a_mask.sum() == 0:
            continue
        a_scores = y_score[a_mask]
        yt_a = np.concatenate([np.zeros(len(h_scores)), np.ones(len(a_scores))])
        ys_a = np.concatenate([h_scores, a_scores])
        au_a = roc_auc_score(yt_a, ys_a)
        anom_scores[(test_task, anom)] = (yt_a, ys_a)
        print(f"    {test_task} {anom}: AUROC={au_a:.4f}")

# STEP 4 -- Bootstrap BCa CIs
print(f"\n[Step 4] Bootstrap BCa CIs (N_BOOT={N_BOOT})...")
rng = np.random.default_rng(42)

agg_ci_rows = []
for task in TASKS:
    yt, ys = agg_scores[task]
    auroc, lo, hi = bootstrap_auroc_bca(yt, ys, rng=rng)
    agg_ci_rows.append(dict(
        test_task=task, method="Conv-AE (raw current)",
        n_healthy=int((yt==0).sum()), n_anomaly=int((yt==1).sum()),
        auroc=round(auroc,4), ci_lo=round(lo,4), ci_hi=round(hi,4),
        ci_width=round(hi-lo,4)))
    print(f"  {task}: AUROC={auroc:.4f} [{lo:.4f}, {hi:.4f}]")

pd.DataFrame(agg_ci_rows).to_csv(
    os.path.join(OUT, "NB10bd_convae_auroc_aggregate.csv"), index=False)
print(f"  Saved: NB10bd_convae_auroc_aggregate.csv")

anom_ci_rows = []
for (task, anom), (yt, ys) in anom_scores.items():
    auroc, lo, hi = bootstrap_auroc_bca(yt, ys, rng=rng)
    anom_ci_rows.append(dict(
        test_task=task, anomaly_type=anom, method="Conv-AE (raw current)",
        n_healthy=int((yt==0).sum()), n_anomaly=int((yt==1).sum()),
        auroc=round(auroc,4), ci_lo=round(lo,4), ci_hi=round(hi,4),
        ci_width=round(hi-lo,4)))

pd.DataFrame(anom_ci_rows).sort_values(["anomaly_type","test_task"]).to_csv(
    os.path.join(OUT, "NB10bd_convae_auroc_per_anomaly.csv"), index=False)
print(f"  Saved: NB10bd_convae_auroc_per_anomaly.csv")

print("\nNB10bd complete.")
print(f"Outputs: {OUT}/NB10bd_convae_*.csv")
print("\nT1/T2/T3 numbers should reproduce NB10b_convae_auroc_aggregate.csv byte-for-byte.")
print("T4 row is new.")